# 04 · Graph Convolutional Networks (GCN)

## What this notebook covers
GCN (Kipf & Welling, 2017) is the seminal spectral graph convolutional network. This notebook covers the spectral motivation at a high level and implements GCN with PyTorch Geometric.

## Spectral motivation (narrative)
The theoretical foundation of GCN comes from graph signal processing. A graph signal is a function mapping nodes to values (node features). The graph Laplacian L = D - A acts as a discrete analogue of the Laplace operator. Just as CNNs perform convolution in frequency space (via FFT), we can define convolution on graphs in the eigenbasis of L.

**ChebNet** (Defferrard et al. 2016) approximated graph convolutions using Chebyshev polynomials to avoid computing the full eigenbasis. GCN (2017) simplified this to a first-order approximation with a renormalisation trick, giving the clean update rule:

    H' = σ(D̃^{-1/2} Ã D̃^{-1/2} H W)

where Ã = A + I (add self-loops) and D̃ is the corresponding degree matrix.

**Why the spectral framing matters less today**: spectral GCNs are limited to fixed graph structure (transductive). Spatial/message-passing methods (GraphSAGE, GAT, GIN) work inductively and are more widely used. GCN is still a strong baseline and widely deployed in production.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

try:
    from torch_geometric.datasets import Planetoid, KarateClub
    from torch_geometric.nn import GCNConv
    from torch_geometric.utils import to_networkx
    HAS_PYG = True
    print("PyTorch Geometric available")
except ImportError:
    HAS_PYG = False
    print("PyTorch Geometric not installed. Install with:")
    print("  pip install torch-geometric")
    print("\nFalling back to manual GCN implementation.")
torch.manual_seed(42); np.random.seed(42)

In [ ]:
if HAS_PYG:
    # ── GCN on Cora citation network ────────────────────────────────────────────
    dataset = Planetoid(root="/tmp/Cora", name="Cora")
    data = dataset[0]
    print(f"Cora: {data.num_nodes} nodes  {data.num_edges} edges  {dataset.num_classes} classes")
    print(f"Node feature dim: {data.num_features}")
    print(f"Train/Val/Test: {data.train_mask.sum()}/{data.val_mask.sum()}/{data.test_mask.sum()}")

    class GCN(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, out_channels)

        def forward(self, x, edge_index):
            x = F.relu(self.conv1(x, edge_index))
            x = F.dropout(x, p=0.5, training=self.training)
            return self.conv2(x, edge_index)

    model = GCN(dataset.num_features, 64, dataset.num_classes)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    def train():
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        F.cross_entropy(out[data.train_mask], data.y[data.train_mask]).backward()
        opt.step()

    def test():
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            accs = []
            for mask in [data.train_mask, data.val_mask, data.test_mask]:
                accs.append((out.argmax(dim=1)[mask] == data.y[mask]).float().mean().item())
        return accs

    best_val, best_test = 0, 0
    for epoch in range(200):
        train()
        tr, val, te = test()
        if val > best_val:
            best_val, best_test = val, te
    print(f"GCN on Cora: Test Accuracy = {best_test:.4f}  (best val = {best_val:.4f})")
else:
    print("Skipping Cora — PyTorch Geometric not available.")
    print("GCN on Cora typically achieves ~81% test accuracy.")

In [ ]:
# ── Manual GCN on Karate Club (fallback + illustration) ──────────────────────
import networkx as nx
import torch.nn as nn

G_kc = nx.karate_club_graph()
n = G_kc.number_of_nodes()
A = nx.to_numpy_array(G_kc) + np.eye(n)
D_inv_sqrt = np.diag(1/np.sqrt(A.sum(1)))
A_hat = torch.FloatTensor(D_inv_sqrt @ A @ D_inv_sqrt)
X_kc  = torch.eye(n)
y_kc  = torch.LongTensor([0 if G_kc.nodes[i]["club"]=="Mr. Hi" else 1 for i in range(n)])

class ManualGCN(nn.Module):
    def __init__(self, in_d, hid_d, out_d):
        super().__init__()
        self.W1 = nn.Linear(in_d, hid_d, bias=False)
        self.W2 = nn.Linear(hid_d, out_d, bias=False)
    def forward(self, X, A):
        H = F.relu(self.W1(A @ X))
        return self.W2(A @ H)

gcn = ManualGCN(n, 16, 2)
opt = torch.optim.Adam(gcn.parameters(), lr=0.01)
for _ in range(300):
    out = gcn(X_kc, A_hat)
    loss = F.cross_entropy(out, y_kc)
    opt.zero_grad(); loss.backward(); opt.step()

gcn.eval()
with torch.no_grad():
    preds = gcn(X_kc, A_hat).argmax(dim=1)
acc = (preds == y_kc).float().mean().item()
print(f"Manual GCN on Karate Club accuracy: {acc:.3f}")

In [ ]:
# Visualise GCN node classification
from sklearn.manifold import TSNE
gcn.eval()
with torch.no_grad():
    h = F.relu(gcn.W1(A_hat @ X_kc)).numpy()

h_2d = TSNE(n_components=2, perplexity=5, random_state=0).fit_transform(h)
colours = ["steelblue" if yi==0 else "tomato" for yi in y_kc.numpy()]
pos = nx.spring_layout(G_kc, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(h_2d[:,0], h_2d[:,1], c=colours, s=80, alpha=0.8)
for i in range(n): axes[0].annotate(str(i), h_2d[i], fontsize=6, alpha=0.5)
axes[0].set_title("GCN hidden layer embeddings (t-SNE)")

c_pred = ["steelblue" if p==0 else "tomato" for p in preds.numpy()]
nx.draw(G_kc, pos=pos, ax=axes[1], node_color=c_pred, with_labels=True, font_size=7, node_size=200, edge_color="grey", alpha=0.8)
axes[1].set_title(f"GCN predictions (acc={acc:.2f})")
plt.tight_layout()
plt.savefig(f"{base}/04_gcn.png", dpi=120)
plt.show()